# 실험 3. LightGBM 5개 시드 앙상블

실험 기록의 최종 조합 R3와 min_child_samples=10 설정으로 LightGBM을 다섯 번 학습하고 예측을 평균합니다. 연습 데이터의 검증 점수를 함께 보고 제출 형식의 CSV를 저장합니다.

검증은 학습 데이터 날짜를 하루씩 제외하는 방식으로 진행합니다. 무작위 분할은 사용하지 않습니다.

## 1. 환경 준비

Python 3.12 커널을 선택한 뒤 필요하면 아래 설치 셀을 한 번 실행하고 커널을 재시작합니다.

In [ ]:
import sys
assert (3, 12) <= sys.version_info[:2] <= (3, 14), f"Python 3.12 환경을 선택하세요. 현재: {sys.version.split()[0]}"
%pip install -r ../requirements.txt

## 2. 데이터 불러오기

처음에는 연습용 데이터(practice)로 실행하고, 대회 때는 PHASE를 inal로 변경합니다.

In [ ]:
from pathlib import Path
import pandas as pd

PHASE = 'practice'  # 'practice' 또는 'final'
REPO_ID = {'practice': 'CAMUS-LAB/bis-travel-time-practice',
           'final': 'CAMUS-LAB/bis-travel-time-final'}[PHASE]
DATA = Path('../data') / PHASE
if not (DATA / 'train.parquet').exists():
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=DATA)

train = pd.read_parquet(DATA / 'train.parquet')
test = pd.read_parquet(DATA / 'test.parquet')
train['date'] = train['date'].astype(str)
print('train:', train.shape, '| test:', test.shape)
print('train dates:', sorted(train.date.unique()))
print('test labels available:', 'tt_sec' in test.columns)

## 3. 최종 특성 조합 R3

배차간격과 is_peak는 제외하고 출발·도착 정류장 ID를 포함합니다. 이는 실험 기록에서 선택된 조합입니다.

In [ ]:
import os
import numpy as np
import lightgbm as lgb

PEAK = {7, 8, 9, 17, 18, 19}
CAT = ['route_id', 'from_stop', 'to_stop']
NUM = ['route_type', 'direction', 'from_lat', 'from_lon', 'to_lat', 'to_lon',
       'dlat', 'dlon', 'stop_seq', 'dist_m', 'arrive_hour', 'is_weekend']

def add_features(df):
    out = df.copy()
    day = pd.to_datetime(out['date']).dt.dayofweek
    out['is_weekend'] = (day >= 5).astype('int8')
    out['dlat'] = out['to_lat'] - out['from_lat']
    out['dlon'] = out['to_lon'] - out['from_lon']
    return out

all_features = add_features(pd.concat([train, test], ignore_index=True))
CATEGORIES = {col: pd.Index(all_features[col].dropna().unique()) for col in CAT}

def prepare(df):
    x = add_features(df)[CAT + NUM].copy()
    for col in CAT:
        x[col] = pd.Categorical(x[col], categories=CATEGORIES[col])
    return x

def mae(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))

## 4. 날짜 단위 검증

각 날짜를 한 번씩 검증용으로 제외합니다. 학습일이 평일/토요일 어느 쪽인지에 따라 결과를 따로 집계합니다.

In [ ]:
fold_rows = []
for val_date in sorted(train.date.unique()):
    fit = train[train.date != val_date]
    val = train[train.date == val_date]
    pred = np.zeros(len(val), dtype=float)
    for seed in range(5):
        model = lgb.LGBMRegressor(
            objective='regression_l1', learning_rate=0.05, num_leaves=127,
            min_child_samples=10, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, n_estimators=1055, random_state=seed, verbose=-1)
        model.fit(prepare(fit), fit['tt_sec'])
        pred += model.predict(prepare(val)) / 5
    fold_rows.append({'date': val_date, 'MAE': mae(val['tt_sec'], pred),
                      'weekday': pd.Timestamp(val_date).dayofweek < 5})

fold_scores = pd.DataFrame(fold_rows)
weekday_mae = fold_scores.loc[fold_scores['weekday'], 'MAE'].mean()
weekend_mae = fold_scores.loc[~fold_scores['weekday'], 'MAE'].mean()
print(f'평일 MAE: {weekday_mae:.3f}')
print(f'토요일 MAE: {weekend_mae:.3f}')
print(f'4일 평균 MAE: {(3 * weekday_mae + weekend_mae) / 4:.3f}')
fold_scores

## 5. 전체 학습 및 제출 파일 저장

검증 결과 확인 후 전체 학습 데이터로 다섯 개 모델을 다시 학습합니다. practice test에는 정답이 있어 채점하고, final test에는 정답이 없으므로 CSV 형식만 점검합니다.

In [ ]:
final_pred = np.zeros(len(test), dtype=float)
for seed in range(5):
    model = lgb.LGBMRegressor(
        objective='regression_l1', learning_rate=0.05, num_leaves=127,
        min_child_samples=10, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, n_estimators=1055, random_state=seed, verbose=-1)
    model.fit(prepare(train), train['tt_sec'])
    final_pred += model.predict(prepare(test)) / 5

submission = pd.DataFrame({'row_id': test['row_id'], 'tt_sec': np.round(final_pred, 1)})
assert submission['row_id'].notna().all()
assert submission['row_id'].is_unique
assert submission['tt_sec'].notna().all()
assert (submission['tt_sec'] > 0).all()

os.makedirs('../submissions', exist_ok=True)
out_path = f'../submissions/{PHASE}_lightgbm_seed5.csv'
submission.to_csv(out_path, index=False)
print(f'저장: {out_path} | 행: {len(submission):,}')

if 'tt_sec' in test.columns:
    import sys
    sys.path.append('../src')
    from evaluate import score
    report = score(submission, test[['row_id', 'tt_sec']], test)
    print(f"MAE {report['MAE']:.2f} | RMSE {report['RMSE']:.2f}")